In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

from glob import glob
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 14})

In [ ]:
city_name = 'SC_JOINVILLE'

obs = pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv', parse_dates=['date_first_symptoms'])

obs['month'] = obs['date_first_symptoms'].dt.month
obs['year'] = obs['date_first_symptoms'].apply(lambda x: x.year + 1 if x.month >= 8 else x.year)

cases_2024 = obs[obs['year'] == 2024].groupby(['region', 'city_residency']).agg({'n_cases': 'sum', 'population': 'mean'}).reset_index()
cases_2024['dengue_inc'] = cases_2024['n_cases'] / cases_2024['population'] * 100000
cases_2024['log_cases'] = np.log10(cases_2024['n_cases'] + 1)

gdf = gpd.read_file('utils/municipios/municipios.shp')
gdf['city_name'] = gdf['SIGLA_UF'] + '_' + gdf['city_name']

cases_2024 = gdf.set_index(['city_name']).join(cases_2024.set_index('city_residency')[['dengue_inc', 'log_cases', 'n_cases']])

In [ ]:
ext_input = pd.read_csv('data/ext_porto_alegre.csv', parse_dates=['date_first_symptoms'])
nat_input = pd.read_csv('data/nat_porto_alegre.csv', parse_dates=['date_first_symptoms'])

In [ ]:
ext_cases = pd.concat([
    pd.read_csv(ifile, parse_dates=['date_first_symptoms']) for ifile in glob('data/sensitivity-cities-immunity/act/*.csv')
])
nat_cases = pd.concat([
    pd.read_csv(ifile, parse_dates=['date_first_symptoms']) for ifile in glob('data/sensitivity-cities-immunity/nat/*.csv')
])
act_imm = pd.concat([
    pd.read_csv(ifile, parse_dates=['date_first_symptoms']) for ifile in glob('data/sensitivity-cities-immunity/act-noimmunity/*.csv')
]) 

ext_cases

In [ ]:
obs_quantiles = obs[(obs['city_residency'] == city_name) & (obs['year'] < 2024)]
obs_quantiles = obs_quantiles.pivot_table(index='month', columns='year', values='dengue_inc').quantile([0.025, 0.5, 0.975], axis=1).transpose()

obs_quantiles['date_first_symptoms'] = obs_quantiles.index.map(lambda x: f'2023-{x:02d}-01' if x >= 8 else f'2024-{x:02d}-01')
obs_quantiles['date_first_symptoms'] = pd.to_datetime(obs_quantiles['date_first_symptoms'])
obs_quantiles = obs_quantiles.sort_values('date_first_symptoms')
obs_quantiles = obs_quantiles.reset_index().set_index('date_first_symptoms')

obs_quantiles

In [ ]:
from matplotlib.gridspec import GridSpec
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig = plt.figure(figsize=(18, 5))
gs = GridSpec(1, 3, figure=fig, hspace=0.45, wspace=0.2, width_ratios=[2, 1, 1])

ax_map    = fig.add_subplot(gs[0])
ax_cases  = fig.add_subplot(gs[1])
ax_temp   = fig.add_subplot(gs[2])

dates = pd.date_range('2023-09-01', '2024-07-01', freq='3MS')

def set_xticks(ax):
    ax.set_xticks(dates)
    ax.set_xticklabels([date.strftime('%b\n%Y') for date in dates])
    ax.xaxis.set_minor_locator(plt.NullLocator())

# --- Panel 1: Map ---
ax = ax_map

region = gpd.read_file('utils/BR_UF_2022/BR_UF_2022.shp')
world = gpd.read_file('utils/ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp')

world.plot(ax=ax, color='lightgrey', edgecolor='black', lw=0.5)
cases_2024.plot(
    ax=ax,
    column='log_cases',
    vmin=2, vmax=6,
    cmap='Purples',
    legend=False
)
region.plot(ax=ax, color='none', edgecolor='black', lw=1)
ax.scatter(
    [gdf[gdf['city_name'] == city_name].centroid.x],
    [gdf[gdf['city_name'] == city_name].centroid.y],
    color='red', marker='*'
)
ax.text(
    gdf[gdf['city_name'] == city_name].centroid.x - 2,
    gdf[gdf['city_name'] == city_name].centroid.y - 0.8,
    'Joinville'
)
ax.set_xlim([-58, -48])
ax.set_ylim([-30, -22])

# Colorbar placed manually below ax_map after tight_layout
sm = cm.ScalarMappable(cmap='Purples', norm=mcolors.Normalize(vmin=2, vmax=6))
sm.set_array([])

# --- Panel 2: Dengue Comparison ---
ax = ax_cases

ext_subset = ext_cases[ext_cases['city_residency'] == city_name]
nat_subset = nat_cases[nat_cases['city_residency'] == city_name]
imm_subset = act_imm[act_imm['city_residency'] == city_name]

ext_quantiles = ext_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='predicted_incidence'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()
nat_quantiles = nat_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='predicted_incidence'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()
imm_quantiles = imm_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='predicted_incidence'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()

obs_cases = ext_subset[ext_subset['ensemble_member'] == 0][
    ['date_first_symptoms', 'actual_incidence']
].set_index('date_first_symptoms')

obs_cases['actual_incidence'].plot(ax=ax, color='k', lw=2)
ext_quantiles[0.500].plot(ax=ax, color='C1')
ax.fill_between(
    ext_quantiles.index, ext_quantiles[0.025], ext_quantiles[0.975], alpha=0.25, color='C1'
)
nat_quantiles[0.500].plot(ax=ax, color='C0')
ax.fill_between(
    nat_quantiles.index, nat_quantiles[0.025], nat_quantiles[0.975], alpha=0.25, color='C0'
)
imm_quantiles[0.500].plot(ax=ax, color='C2')
ax.fill_between(
    imm_quantiles.index, imm_quantiles[0.025], imm_quantiles[0.975], alpha=0.25, color='C2'
)

imm = obs[(obs['city_residency'] == city_name) & (obs['year'] == 2024)][['Pr_0priorinf', 'Pr_1priorinf']].iloc[0]
ax.text(
    0.01, 
    0.92, 
    f"Proportion never infected: {int(imm['Pr_0priorinf'] * 100):02d}%\nProportion infected once: {int(imm['Pr_1priorinf'] * 100):02d}%", 
    transform=ax.transAxes,
    fontsize=10
)

ax.set_xlabel('Date')
ax.set_ylabel('Dengue Incidence\n(cases per 100,000)')
ax.set_ylim(0, 7500)
set_xticks(ax)

# --- Panel 3: Temperature ---
ax = ax_temp

ext_subset = ext_input[ext_input['city_residency'] == city_name]
nat_subset = nat_input[nat_input['city_residency'] == city_name]

ext_quantiles = ext_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='mean_2m_air_temp_degree1'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()
nat_quantiles = nat_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='mean_2m_air_temp_degree1'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()

ext_quantiles[0.500].plot(ax=ax, color='C1')
ax.fill_between(
    ext_quantiles.index, ext_quantiles[0.025], ext_quantiles[0.975], alpha=0.25, color='C1'
)
nat_quantiles[0.500].plot(ax=ax, color='C0')
ax.fill_between(
    nat_quantiles.index, nat_quantiles[0.025], nat_quantiles[0.975], alpha=0.25, color='C0'
)
obs_temp = obs[(obs['city_residency'] == city_name) & (obs['date_first_symptoms'] >= '2023-08-01')]
obs_temp.set_index('date_first_symptoms').plot(
    y='mean_2m_air_temp_degree1', color='k', lw=2, ax=ax, legend=False
)

spans = [
    [0.6, 22.9, 30, '#737373'],
    [0.8, 24.4, 27.4, '#404040'],
    [0.9, 25.7, 26.2, '#000000']
]

for span in spans:
    ax.axhspan(span[1], span[2], color=span[3], alpha=0.2)

ax.set_xlabel('Date')
ax.set_ylabel(r'Temperature ($^o$C)')
ax.set_ylim(12, 30)
set_xticks(ax)

# legend
legend_elements = [
    Line2D([0], [0], color='k', lw=2, label='Observed'),
    Patch(facecolor='C1', alpha=0.25, label='ACT (C15-c(PC,SR,S,IM))'),
    Patch(facecolor='C0', alpha=0.25, label='NAT (C15-c(PC,SR,S,IM))'),
    Patch(facecolor='C2', alpha=0.25, label='ACT (C15-c(PC,SR,S))'),
    Patch(facecolor='#737373', alpha=0.5, label='Temp Coef. > P60'),
    Patch(facecolor='#404040', alpha=0.5, label='Temp Coef. > P80'),
    Patch(facecolor='#000000', alpha=0.5, label='Temp Coef. > P90')
]

fig.legend(
    handles=legend_elements,
    loc='upper right',
    bbox_to_anchor=(0.85, -0.1),
    frameon=False,
    ncols=4
)

# --- Add colorbar after layout is finalised ---
plt.tight_layout()

# Read ax_map position and place colorbar just below it
pos = ax_map.get_position()
cax = fig.add_axes([pos.x0, pos.y0 - 0.1, pos.width * 1, 0.04])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label('log(Dengue Cases)')
plt.savefig('img/joinville.png', bbox_inches='tight', dpi=300)

In [ ]:
city_name = 'BA_ILHEUS'

obs = pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv', parse_dates=['date_first_symptoms'])

obs['month'] = obs['date_first_symptoms'].dt.month
obs['year'] = obs['date_first_symptoms'].apply(lambda x: x.year + 1 if x.month >= 8 else x.year)

cases_2024 = obs[obs['year'] == 2024].groupby(['region', 'city_residency']).agg({'n_cases': 'sum', 'population': 'mean'}).reset_index()
cases_2024['dengue_inc'] = cases_2024['n_cases'] / cases_2024['population'] * 100000
cases_2024['log_cases'] = np.log10(cases_2024['n_cases'] + 1)

gdf = gpd.read_file('utils/municipios/municipios.shp')
gdf['city_name'] = gdf['SIGLA_UF'] + '_' + gdf['city_name']

cases_2024 = gdf.set_index(['city_name']).join(cases_2024.set_index('city_residency')[['dengue_inc', 'log_cases', 'n_cases']])

In [ ]:
ext_input = pd.read_csv('data/ext_ilheus.csv', parse_dates=['date_first_symptoms'])
nat_input = pd.read_csv('data/nat_ilheus.csv', parse_dates=['date_first_symptoms'])

In [ ]:
ext_cases = pd.concat([
    pd.read_csv(ifile, parse_dates=['date_first_symptoms']) for ifile in glob('data/sensitivity-cities-immunity/act/*.csv')
])
nat_cases = pd.concat([
    pd.read_csv(ifile, parse_dates=['date_first_symptoms']) for ifile in glob('data/sensitivity-cities-immunity/nat/*.csv')
])
act_imm = pd.concat([
    pd.read_csv(ifile, parse_dates=['date_first_symptoms']) for ifile in glob('data/sensitivity-cities-immunity/act-noimmunity/*.csv')
]) 

ext_cases

In [ ]:
obs_quantiles = obs[(obs['city_residency'] == city_name) & (obs['year'] < 2024)]
obs_quantiles = obs_quantiles.pivot_table(index='month', columns='year', values='dengue_inc').quantile([0.025, 0.5, 0.975], axis=1).transpose()

obs_quantiles['date_first_symptoms'] = obs_quantiles.index.map(lambda x: f'2023-{x:02d}-01' if x >= 8 else f'2024-{x:02d}-01')
obs_quantiles['date_first_symptoms'] = pd.to_datetime(obs_quantiles['date_first_symptoms'])
obs_quantiles = obs_quantiles.sort_values('date_first_symptoms')
obs_quantiles = obs_quantiles.reset_index().set_index('date_first_symptoms')

obs_quantiles

In [ ]:
from matplotlib.gridspec import GridSpec
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig = plt.figure(figsize=(18, 5))
gs = GridSpec(1, 3, figure=fig, hspace=0.45, wspace=0.2, width_ratios=[2, 1, 1])

ax_map   = fig.add_subplot(gs[0])
ax_cases = fig.add_subplot(gs[1])
ax_temp  = fig.add_subplot(gs[2])

dates = pd.date_range('2023-09-01', '2024-07-01', freq='3MS')

def set_xticks(ax):
    ax.set_xticks(dates)
    ax.set_xticklabels([date.strftime('%b\n%Y') for date in dates])
    ax.xaxis.set_minor_locator(plt.NullLocator())

# --- Panel 1: Map ---
ax = ax_map

region = gpd.read_file('utils/BR_UF_2022/BR_UF_2022.shp')
world = gpd.read_file('utils/ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp')

world.plot(ax=ax, color='lightgrey', edgecolor='black', lw=0.5)
cases_2024.plot(
    ax=ax,
    column='log_cases',
    vmin=2, vmax=6,
    cmap='Purples',
    legend=False
)
region.plot(ax=ax, color='none', edgecolor='black', lw=1)
ax.scatter(
    [gdf[gdf['city_name'] == city_name].centroid.x],
    [gdf[gdf['city_name'] == city_name].centroid.y],
    color='red', marker='*'
)
ax.text(
    gdf[gdf['city_name'] == city_name].centroid.x - 1,
    gdf[gdf['city_name'] == city_name].centroid.y - 0.8,
    'Ilhéus'
)
ax.set_xlim([-46, -36])
ax.set_ylim([-19, -11])

sm = cm.ScalarMappable(cmap='Purples', norm=mcolors.Normalize(vmin=2, vmax=6))
sm.set_array([])

# --- Panel 2: Dengue Comparison ---
ax = ax_cases

ext_subset = ext_cases[ext_cases['city_residency'] == city_name]
nat_subset = nat_cases[nat_cases['city_residency'] == city_name]
imm_act_subset = act_imm[act_imm['city_residency'] == city_name]

ext_quantiles = ext_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='predicted_incidence'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()
nat_quantiles = nat_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='predicted_incidence'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()
imm_act_quantiles = imm_act_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='predicted_incidence'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()

obs_cases = ext_subset[ext_subset['ensemble_member'] == 0][
    ['date_first_symptoms', 'actual_incidence']
].set_index('date_first_symptoms')

obs_cases['actual_incidence'].plot(ax=ax, color='k', lw=2)
ext_quantiles[0.500].plot(ax=ax, color='C1')
ax.fill_between(
    ext_quantiles.index, ext_quantiles[0.025], ext_quantiles[0.975], alpha=0.25, color='C1'
)
nat_quantiles[0.500].plot(ax=ax, color='C0')
ax.fill_between(
    nat_quantiles.index, nat_quantiles[0.025], nat_quantiles[0.975], alpha=0.25, color='C0'
)
imm_act_quantiles[0.500].plot(ax=ax, color='C2')
ax.fill_between(
    imm_act_quantiles.index, imm_act_quantiles[0.025], imm_act_quantiles[0.975], alpha=0.25, color='C2'
)

imm = obs[(obs['city_residency'] == city_name) & (obs['year'] == 2024)][['Pr_0priorinf', 'Pr_1priorinf']].iloc[0]
ax.text(
    0.01, 
    0.92, 
    f"Proportion never infected: {int(imm['Pr_0priorinf'] * 100):02d}%\nProportion infected once: {int(imm['Pr_1priorinf'] * 100):02d}%", 
    transform=ax.transAxes,
    fontsize=10
)

ax.set_xlabel('Date')
ax.set_ylabel('Dengue Incidence\n(cases per 100,000)')
ax.set_ylim(0, 700)
set_xticks(ax)

# --- Panel 3: Temperature ---
ax = ax_temp

ext_subset = ext_input[ext_input['city_residency'] == city_name]
nat_subset = nat_input[nat_input['city_residency'] == city_name]

ext_quantiles = ext_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='mean_2m_air_temp_degree1'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()
nat_quantiles = nat_subset.pivot_table(
    index='date_first_symptoms', columns='ensemble_member', values='mean_2m_air_temp_degree1'
).quantile([0.025, 0.5, 0.975], axis=1).transpose()

obs_cases = ext_subset[ext_subset['ensemble_member'] == 0][
    ['date_first_symptoms', 'mean_2m_air_temp_degree1']
].set_index('date_first_symptoms')

ext_quantiles[0.500].plot(ax=ax, color='C1')
ax.fill_between(
    ext_quantiles.index, ext_quantiles[0.025], ext_quantiles[0.975], alpha=0.25, color='C1'
)
nat_quantiles[0.500].plot(ax=ax, color='C0')
ax.fill_between(
    nat_quantiles.index, nat_quantiles[0.025], nat_quantiles[0.975], alpha=0.25, color='C0'
)

obs_temp = obs[(obs['city_residency'] == city_name) & (obs['date_first_symptoms'] >= '2023-08-01')]
obs_temp.set_index('date_first_symptoms').plot(
    y='mean_2m_air_temp_degree1', color='k', lw=2, ax=ax, legend=False
)

spans = [
    [0.6, 22.9, 30, '#737373'],
    [0.8, 24.4, 27.4, '#404040'],
    [0.9, 25.7, 26.2, '#000000']
]

for span in spans:
    ax.axhspan(span[1], span[2], color=span[3], alpha=0.2)

ax.set_xlabel('Date')
ax.set_ylabel(r'Temperature ($^o$C)')
ax.set_ylim(12, 30)
set_xticks(ax)

# --- Finalise ---
plt.tight_layout()

pos = ax_map.get_position()
cax = fig.add_axes([pos.x0, pos.y0 - 0.1, pos.width * 1, 0.04])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label('log(Dengue Cases)')
plt.savefig('img/ilheus.png', bbox_inches='tight', dpi=300)